To add: 
- notebook header
- short introduction
- table of contents

This notebook shows how to train a custom crop type model using private reference data.

Before you start running this notebook, you need to make sure your private dataset(s) are uploaded to the RDM...

### 1. Run satellite data extractions for your private datasets

< explanation >

Check which private collections exist:

In [ ]:
from worldcereal.rdm_api import RdmInteraction

rdm = RdmInteraction()
collections = rdm.get_collections(include_public=False,
                                  include_private=True)
for collection in collections:
    print('----------')
    print(f'ID: {collection.id}')
    print(f'Title: {collection.title}')
    print(f'Sample count: {collection.feature_count}')

Now download all samples for your collection of interest:

In [ ]:
collection_id = input("Enter the collection id for which you would like to start extractions: ")

gdf = rdm.download_samples(ref_ids=[collection_id])
gdf.head()

Define a folder where extractions will be stored

In [ ]:
from pathlib import Path

extractions_dir = Path('/vitodata/worldcereal/tmp/jeroen/point_extractions')
extractions_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# save samples to geoparquet
outfolder_col = extractions_dir / collection_id
outfolder_col.mkdir(parents=True, exist_ok=True)
df_file = str(outfolder_col / 'dataframe.geoparquet')
gdf.to_parquet(df_file)

Now we fire up point extractions for all downloaded samples.

**Note that extractions should be launched separately for each individual collection_id**

In [ ]:
# TODO: start and end date of extractions is set based on range of valid_time in individual jobs (all samples in same year are grouped together)!!
# Should we group every 4 months instead of per year, to avoid having excessive lengths of time series?

In [ ]:
from worldcereal.stac.constants import ExtractionCollection

# TODO: we need a better way to call the extractions script!
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions')
from extract import run_extractions

collection_type = ExtractionCollection('POINT_WORLDCEREAL')
run_extractions(collection_type, outfolder_col, Path(df_file))

### 2. Define your region of interest

When running the code snippet below, an interactive map will be visualized.
Click the Rectangle button on the left hand side of the map to start drawing your region of interest.
The widget will automatically store the coordinates of the last rectangle you drew on the map.

<div class="alert alert-block alert-warning">
<b>Processing area limitation:</b><br> 
Processing areas beyond 2500 km² are currently not supported to avoid excessive credit usage and long processing times.<br>
Upon exceeding this limit, an error will be shown, and you will need to draw a new rectangle.

For testing purposes, we recommend you to select a small area (< 250 km²) in order to limit processing time and credit usage.

A run of 250 km² will typically consume 40 credits and last around 20 mins.<br>
A run of 750 km² will typically consume 90 credits and last around 50 mins.<br>
A run of 2500 km² will typically consume 250 credits and last around 1h 40 mins.
</div>

In [ ]:
from worldcereal.utils.map import ui_map

map = ui_map()
map.show_map()

### 3. Define your temporal extent

To determine your season of interest, you can consult the WorldCereal crop calendars (by executing the next cell), or check out the [USDA crop calendars](https://ipad.fas.usda.gov/ogamaps/cropcalendar.aspx).

In [ ]:
from utils import retrieve_worldcereal_seasons

spatial_extent = map.get_processing_extent()
seasons = retrieve_worldcereal_seasons(spatial_extent)

Now use the slider to select your processing period. Note that the length of the period is always fixed to a year.
Just make sure your season of interest is fully captured within the period you select.

In [ ]:
from utils import date_slider

slider = date_slider()
slider.show_slider()

### 4. Retrieve and prepare relevant training data

Now we query our local extractions database and retrieve the relevant samples based on defined area of interest and processing period.

You can also include training data from public datasets by using the "include_public" flag.

In [ ]:
from worldcereal.utils.refdata import query_extractions

# Retrieve the polygon you just drew
polygon = map.get_polygon_latlon()

# Retrieve the date range you just selected
processing_period = slider.get_processing_period()

# Define whether you want to include public data or not
# TODO: Setting this to "True" won't work at the moment!
include_public = False

extract_df = query_extractions(polygon,
                               processing_period=processing_period,
                               include_public=include_public,
                               private_extraction_dir=extractions_dir,)
extract_df.head()

In [ ]:
# You can inspect the timeseries for a specific point or points:

from pathlib import Path
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions/point_extractions')
from extract_point_worldcereal import visualize_timeseries

# By default this generates a plot of NDVI and first point in the dataframe
outfile = outfolder_col / 'timeseries_plot.png'
visualize_timeseries(extract_df, outfile)


In [ ]:
# this is a temporary solution which just loads all private extractions

from pathlib import Path
import sys
sys.path.append('/data/users/Private/jeroendegerickx/git/worldcereal/worldcereal-classification/scripts/extractions/point_extractions')
from extract_point_worldcereal import load_point_extractions

extract_df = load_point_extractions(outfolder_col)
print(extract_df.head())


In [ ]:
# most of the following code is just there temporarily, in order to be able to use the process_parquet function as it was implemented for phase I extractions
# TODO: process_parquet should be adapted to work with the new format of the extractions

# Convert extraction dataframe into the format that is required for model training
import numpy as np
import pandas as pd

# create label columns
extract_df['CROPTYPE_LABEL'] = extract_df['ewoc_code'].values
extract_df['LANDCOVER_LABEL'] = np.zeros(extract_df.shape[0])
extract_df['POTAPOV-LABEL-10m'] = np.zeros(extract_df.shape[0])
extract_df['WORLDCOVER-LABEL-10m'] = np.zeros(extract_df.shape[0])

# create DEM columns
extract_df.rename(columns={'elevation': 'DEM-alt-20m'}, inplace=True)
extract_df.rename(columns={'slope': 'DEM-slo-20m'}, inplace=True)

# rename valid_time to valid_date
extract_df['valid_time'] = pd.to_datetime(extract_df['valid_time'])
extract_df.rename(columns={'valid_time': 'valid_date'}, inplace=True)

# others
extract_df['location_id'] = extract_df['sample_id'].values
extract_df['aez_zoneid'] = np.zeros(extract_df.shape[0])

# here we actually run process_parquet:
from worldcereal.utils.refdata import process_parquet
# Prepare training dataframe
training_df = process_parquet(extract_df)
training_df.head()

Now the user needs to select which crop types to include and which ones to merge...

In [ ]:
from utils import CropTypePicker

croptypepicker = CropTypePicker(training_df, samples_threshold=2)
croptypepicker.display()


In [ ]:
# now apply the crop type picker and generate the new dataframe, only containing the classes of interest
# (contained in attribute "final_class")

# QUESTION: in the previous implementation, all non-selected classes were assigned to other_temporary_crops.
# Should we keep this or just remove the non-selected classes from the dataframe?
# currently the latter has been implemented

new_df = croptypepicker.apply()
new_df['final_class'].value_counts()

In [ ]:
# Prepare training features

from utils import prepare_training_dataframe

training_dataframe = prepare_training_dataframe(new_df, task_type="croptype")
training_dataframe.head()

### 5. Custom model training and deployment

In [ ]:
# Train model

from utils import train_classifier

custom_model, report, confusion_matrix = train_classifier(training_dataframe, balance_classes=False)
# Print the classification report
print(report)

In [ ]:
# deploy model

from worldcereal.utils.upload import deploy_model
from openeo_gfmap.backend import cdse_connection
from utils import get_input

modelname = get_input("model")
model_url = deploy_model(cdse_connection(), custom_model, pattern=modelname)
print(f"Your model can be downloaded from: {model_url}")

### 6. Run inference and show resulting map

In [ ]:
# Run inference and visualize output...